# 📐 ĐỀ TÀI SỐ 4: ỨNG DỤNG MULTI-AGENT FRAMEWORK KẾT HỢP TOOL USE (SYMPY) CHO SUY LUẬN SỐ ĐA BƯỚC TRONG TOÁN THCS

**Môn học:** MAT3399 - Các chủ đề nâng cao trong AI / AI Tutor  
**Mục tiêu nghiên cứu:** Xây dựng hệ thống tác tử giải toán có kiểm chứng ký hiệu (**Symbolic Verification**), loại bỏ triệt để hiện tượng ảo giác (hallucination) trong suy luận số học đa bước của LLM.

---
### 🌟 Quy trình Tuần hoàn 5 Tác tử:
1. **Parser Agent**: Bóc tách cấu trúc ngôn ngữ tự nhiên tiếng Việt, trích xuất biến số, điều kiện thực tế ($x > 0$), và quan hệ toán học.
2. **Symbolic Agent**: Thiết lập hệ phương trình hình thức và sinh mã Python SymPy chính xác tuyệt đối.
3. **Safe AST Sandbox**: Cô lập thực thi, phân tích cây cú pháp trừu tượng AST để chặn mọi lệnh nguy hiểm (`os`, `sys`, `subprocess`, `eval`...), khống chế timeout 3s bằng multiprocessing.
4. **Self-Correction Agent**: Bắt exception/traceback từ Sandbox, phân tích nguyên nhân và tự động sửa mã (tối đa 3 vòng lặp).
5. **Pedagogical Agent**: Trình bày lời giải sư phạm chuẩn mực 5 bước theo sách giáo khoa Toán THCS Việt Nam kèm định dạng $\LaTeX$ sắc nét.

## 📦 Bước 1: Chuẩn bị Môi trường & Cài đặt Dependencies
Chạy ô dưới đây để thiết lập đường dẫn dự án và cài đặt toàn bộ thư viện cần thiết (`google-genai`, `langgraph`, `sympy`, `streamlit`, `pyngrok`).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd drive/MyDrive/mat3399_final-project

/content/drive/MyDrive/mat3399_final-project


In [4]:
!pip install -q google-genai langgraph langchain sympy streamlit pyngrok requests pydantic datasets pandas

In [ ]:
# 1. Đảm bảo mã nguồn dự án nằm trong PYTHONPATH
import os, sys
from pathlib import Path

cwd = Path.cwd()
if not (cwd / "src").exists():
    if (cwd / "mat3399_final-project" / "src").exists():
        os.chdir(cwd / "mat3399_final-project")
    elif (cwd.parent / "src").exists():
        os.chdir(cwd.parent)

sys.path.insert(0, str(Path.cwd()))
print(f"[*] Thư mục làm việc: {Path.cwd()}")

# 2. Cài đặt các thư viện phụ thuộc từ requirements.txt nếu có, hoặc danh sách chuẩn
!pip install -q google-genai langgraph langchain sympy streamlit pyngrok requests pydantic datasets pandas
print("✓ Đã cài đặt xong toàn bộ dependencies thành công!")


## 🔑 Bước 2: Cấu hình Khóa API (GEMINI_API_KEY & OPENROUTER_API_KEY)
Hệ thống hỗ trợ 2 nhà cung cấp LLM chính:
1. **Google Gemini**: Lấy khóa miễn phí tại [Google AI Studio](https://aistudio.google.com/app/apikey).
2. **OpenRouter (Tùy chọn)**: Sử dụng các mô hình mã nguồn mở như DeepSeek-R1 miễn phí, lấy khóa tại [OpenRouter Keys](https://openrouter.ai/keys).

Bạn có thể dùng tính năng **Colab Secrets** (biểu tượng chìa khóa ở thanh bên trái) hoặc nhập trực tiếp bên dưới.

In [5]:
import os
from getpass import getpass

# 1. Cấu hình GEMINI_API_KEY
try:
    from google.colab import userdata
    gemini_key = userdata.get("GEMINI_API_KEY")
except Exception:
    gemini_key = None

if not gemini_key:
    gemini_key = os.getenv("GEMINI_API_KEY")

if not gemini_key:
    gemini_key = getpass("👉 Nhập GEMINI_API_KEY (lấy tại https://aistudio.google.com/): ")

if gemini_key and gemini_key.strip():
    os.environ["GEMINI_API_KEY"] = gemini_key.strip()
    print("✓ Đã cấu hình GEMINI_API_KEY thành công!")
else:
    print("⚠️ Chưa nhập GEMINI_API_KEY.")

✓ Đã cấu hình GEMINI_API_KEY thành công!


In [ ]:
# 2. Cấu hình OPENROUTER_API_KEY (Tùy chọn cho DeepSeek-R1)
try:
    from google.colab import userdata
    openrouter_key = userdata.get("OPENROUTER_API_KEY")
except Exception:
    openrouter_key = None

if not openrouter_key:
    openrouter_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_key:
    user_input_or = input("👉 (Tùy chọn) Nhập OPENROUTER_API_KEY (Nhấn Enter nếu muốn bỏ qua): ")
    if user_input_or.strip():
        openrouter_key = user_input_or.strip()

if openrouter_key and openrouter_key.strip():
    os.environ["OPENROUTER_API_KEY"] = openrouter_key.strip()
    print("✓ Đã cấu hình OPENROUTER_API_KEY thành công!")
else:
    print("ℹ️ Bỏ qua OPENROUTER_API_KEY (hệ thống sẽ sử dụng Gemini mặc định).")

## 🔍 Bước 2.1: Xem trước & Khám phá Bộ Dữ liệu Chuẩn GSM8K Tiếng Việt (`hllj/vi_gsm8k`)
Bộ dữ liệu chuẩn `hllj/vi_gsm8k` trên Hugging Face bao gồm **1.319 câu hỏi** suy luận số đa bước trong tập `test`. Ô lệnh dưới đây tải dữ liệu trực tiếp vào RAM và hiển thị 5 mẫu đầu tiên qua bảng `pandas` để trực quan hóa câu hỏi, diễn giải và đáp án số học kỳ vọng.

In [6]:
import pandas as pd
from datasets import load_dataset
from IPython.display import display
from benchmark import extract_gsm8k_answer

print("[*] Đang nạp trực tiếp bộ dữ liệu 'hllj/vi_gsm8k' (split='test') từ Hugging Face Hub vào RAM...")
vi_gsm8k_dataset = load_dataset('hllj/vi_gsm8k', split='test')

# Chuẩn bị dữ liệu hiển thị 5 mẫu đầu tiên
preview_rows = []
for i in range(min(5, len(vi_gsm8k_dataset))):
    item = vi_gsm8k_dataset[i]
    q = item.get('question', "")
    ans = item.get('answer', "")
    expected_num = extract_gsm8k_answer(ans)
    preview_rows.append({
        'STT': i + 1,
        'Đề bài (Question)': q,
        'Lời giải chi tiết (Answer)': ans,
        'Đáp số kỳ vọng': expected_num
    })

df_preview = pd.DataFrame(preview_rows)
pd.set_option('display.max_colwidth', None)
display(df_preview)
print(f"\n[✓] Tổng số bài toán trong tập kiểm nghiệm: {len(vi_gsm8k_dataset):,} câu hỏi.")


[*] Đang nạp trực tiếp bộ dữ liệu 'hllj/vi_gsm8k' (split='test') từ Hugging Face Hub vào RAM...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


README.md:   0%|          | 0.00/1.85k [00:00<?, ?B/s]

vi_train.json:   0%|          | 0.00/5.50M [00:00<?, ?B/s]

vi_test.json:   0%|          | 0.00/1.01M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

,STT,Đề bài (Question),Lời giải chi tiết (Answer),Đáp số kỳ vọng
0,1,Vịt của Janet đẻ được 16 quả trứng mỗi ngày. Cô ấy ăn 3 quả vào bữa sáng và nướng bánh muffin cho bạn bè mỗi ngày với 4 quả. Cô ấy bán số trứng còn lại tại chợ nông sản hàng ngày với giá $2 mỗi quả trứng vịt tươi. Cô ấy kiếm được bao nhiêu đô la mỗi ngày tại chợ nông sản?,18,18
1,2,Một chiếc áo choàng cần 2 cuộn sợi màu xanh và một nửa số đó là sợi màu trắng. Tổng cộng cần bao nhiêu cuộn sợi?,3,3
2,3,"Josh quyết định thử sức với việc mua bán nhà. Anh ấy mua một căn nhà với giá $80,000 và sau đó chi thêm $50,000 để sửa chữa. Việc sửa chữa này đã làm tăng giá trị của căn nhà lên 150%. Anh ấy đã kiếm được bao nhiêu lợi nhuận?",70000,70000
3,4,James quyết định chạy 3 cuộc chạy nước rút 3 lần một tuần. Anh ta chạy 60 mét mỗi lần chạy. Tổng số mét anh ta chạy mỗi tuần là bao nhiêu?,540,540
4,5,"Mỗi ngày, Wendi cho mỗi con gà của cô ăn ba cốc thức ăn gà hỗn hợp, bao gồm hạt, sâu mealworm và rau để giúp chúng khỏe mạnh. Cô ấy cho gà ăn trong ba bữa ăn riêng biệt. Vào buổi sáng, cô ấy cho đàn gà của mình 15 cốc thức ăn. Vào buổi chiều, cô ấy cho gà của mình thêm 25 cốc thức ăn. Cô ấy cần cho gà ăn bao nhiêu cốc thức ăn trong bữa ăn cuối cùng của ngày nếu đàn gà của Wendi có 20 con?",20,20



[✓] Tổng số bài toán trong tập kiểm nghiệm: 1,319 câu hỏi.


## 🚀 Bước 3: Thử nghiệm Giải Toán THCS 1 Chạm với Multi-Agent Solver
Thử nghiệm bài toán lập phương trình chuyển động kinh điển trong SGK Toán 9.

In [7]:
# Kiểm tra danh sách model khả dụng với API Key của bạn
from google import genai
import os

client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))
print("Danh sách các mô hình khả dụng:")
for m in client.models.list():
    # Lấy các model hỗ trợ sinh nội dung văn bản
    methods = getattr(m, 'supported_generation_methods', []) or getattr(m, 'supported_actions', [])
    if "generateContent" in methods:
        print("  ✓", m.name)

Danh sách các mô hình khả dụng:
  ✓ models/gemini-2.5-flash
  ✓ models/gemini-2.5-pro
  ✓ models/gemini-2.5-flash-preview-tts
  ✓ models/gemini-2.5-pro-preview-tts
  ✓ models/gemma-4-26b-a4b-it
  ✓ models/gemma-4-31b-it
  ✓ models/gemini-flash-latest
  ✓ models/gemini-flash-lite-latest
  ✓ models/gemini-pro-latest
  ✓ models/gemini-2.5-flash-lite
  ✓ models/gemini-2.5-flash-image
  ✓ models/gemini-3-flash-preview
  ✓ models/gemini-3.1-pro-preview
  ✓ models/gemini-3.1-pro-preview-customtools
  ✓ models/gemini-3.1-flash-lite-preview
  ✓ models/gemini-3.1-flash-lite
  ✓ models/gemini-3-pro-image-preview
  ✓ models/gemini-3-pro-image
  ✓ models/nano-banana-pro-preview
  ✓ models/gemini-3.1-flash-image-preview
  ✓ models/gemini-3.1-flash-image
  ✓ models/gemini-3.1-flash-lite-image
  ✓ models/gemini-nano-banana-2.1
  ✓ models/gemini-3.5-flash
  ✓ models/gemini-3.5-flash-lite
  ✓ models/gemini-omni-flash-preview
  ✓ models/gemini-omni-1.1-flash
  ✓ models/gemini-omni-1.2-flash
  ✓ models/ge

In [8]:
from src.llm.factory import LLMFactory
from src.workflow import MultiAgentMathSolver
from IPython.display import display, Markdown

# Khởi tạo Multi-Agent Solver với Gemini 3.5 Flash (Tự động Fail-fast chống lỗi 503 / 429)
llm_client = LLMFactory.create_client(provider="gemini", model="gemini-3.5-flash")
solver = MultiAgentMathSolver(llm_client=llm_client, timeout=3.0, max_attempts=3)

problem = "Một ô tô đi từ Hà Nội đến Hải Phòng với quãng đường dài 120 km. Khi về, xe đi với vận tốc lớn hơn vận tốc lúc đi là 10 km/h nên thời gian về ít hơn thời gian đi là 24 phút. Tính vận tốc lúc đi của ô tô."

print("=" * 75)
print(f"[*] Đề bài: {problem}")
print("[*] Đang kích hoạt Multi-Agent StateGraph (Tiered Cascade Fail-fast)...")
print("=" * 75)

state = solver.run(problem)

print("")
print(f"[✓] Trạng thái hoàn thành: {state.get('status')}")
print(f"[✓] Số lần tự sửa mã: {state.get('attempt_count')}")
print(f"[✓] Nghiệm được kiểm chứng bởi Sandbox: {state.get('execution_result', {}).get('output')}")

display(Markdown("---"))
display(Markdown("### 🎓 Lời giải Sư phạm chuẩn SGK Toán THCS:"))
display(Markdown(state.get("final_solution", "")))


[*] Đề bài: Một ô tô đi từ Hà Nội đến Hải Phòng với quãng đường dài 120 km. Khi về, xe đi với vận tốc lớn hơn vận tốc lúc đi là 10 km/h nên thời gian về ít hơn thời gian đi là 24 phút. Tính vận tốc lúc đi của ô tô.
[*] Đang kích hoạt Multi-Agent StateGraph (Tiered Cascade Fail-fast)...

[✓] Trạng thái hoàn thành: completed
[✓] Số lần tự sửa mã: 1
[✓] Nghiệm được kiểm chứng bởi Sandbox: 50


---

### 🎓 Lời giải Sư phạm chuẩn SGK Toán THCS:

Chào em! Dưới đây là lời giải chi tiết bài toán theo đúng phương pháp sư phạm của chương trình SGK Toán THCS.

---

### **BÀI GIẢI CHI TIẾT**

Đổi: $24 \text{ phút} = \frac{24}{60} \text{ giờ} = \frac{2}{5} \text{ giờ}$.

#### **Bước 1: Chọn ẩn số và đặt điều kiện thích hợp cho ẩn**
Gọi vận tốc lúc đi của ô tô là $x$ (km/h).  
Điều kiện: $x > 0$.

#### **Bước 2: Biểu diễn các đại lượng chưa biết theo ẩn và các đại lượng đã biết**
* Vận tốc lúc về của ô tô là: $x + 10$ (km/h).
* Thời gian ô tô đi từ Hà Nội đến Hải Phòng là: $\frac{120}{x}$ (giờ).
* Thời gian ô tô đi từ Hải Phòng về Hà Nội là: $\frac{120}{x + 10}$ (giờ).

#### **Bước 3: Lập phương trình**
Vì thời gian lúc về ít hơn thời gian lúc đi là $24$ phút ($\frac{2}{5}$ giờ) nên ta có phương trình:
$$ \frac{120}{x} - \frac{120}{x + 10} = \frac{2}{5} $$

#### **Bước 4: Giải phương trình từng bước**
Quy đồng mẫu thức hai vế với mẫu thức chung là $5x(x + 10)$:

$$ \frac{120 \cdot 5(x + 10)}{5x(x + 10)} - \frac{120 \cdot 5x}{5x(x + 10)} = \frac{2x(x + 10)}{5x(x + 10)} $$

Khử mẫu, ta được phương trình:
$$ 600(x + 10) - 600x = 2x(x + 10) $$
$$ 600x + 6000 - 600x = 2x^2 + 20x $$
$$ 2x^2 + 20x - 6000 = 0 $$

Chia cả hai vế cho $2$, ta có phương trình bậc hai:
$$ x^2 + 10x - 3000 = 0 $$

Giải phương trình bậc hai trên bằng biệt thức $\Delta'$:
* $a = 1$, $b' = 5$, $c = -3000$
* $\Delta' = (b')^2 - ac = 5^2 - 1 \cdot (-3000) = 25 + 3000 = 3025 > 0$
* $\sqrt{\Delta'} = \sqrt{3025} = 55$

Phương trình có hai nghiệm phân biệt:
$$ x_1 = \frac{-b' + \sqrt{\Delta'}}{a} = \frac{-5 + 55}{1} = 50 $$
$$ x_2 = \frac{-b' - \sqrt{\Delta'}}{a} = \frac{-5 - 55}{1} = -60 $$

#### **Bước 5: Đối chiếu nghiệm với điều kiện ban đầu và kết luận bài toán**
* Giá trị $x_1 = 50$ thỏa mãn điều kiện $x > 0$.
* Giá trị $x_2 = -60$ không thỏa mãn điều kiện $x > 0$ (loại).

**Kết luận:** Vậy vận tốc lúc đi của ô tô là **$50 \text{ km/h}$**.

## 📊 Bước 4: Chạy Benchmark Đo lường Định lượng Toàn diện
Đo lường 4 chỉ số khoa học: **Accuracy**, **Execution Success Rate (ESR)**, **Self-Correction Recovery Rate (SCRR)** và **Average Latency**.

Hệ thống cung cấp 2 chế độ đánh giá:
1. **Bộ dữ liệu THCS nội bộ (`thcs_problems.json`)**: Đánh giá các dạng toán lập phương trình SGK (chuyển động, làm chung làm riêng, năng suất...).
2. **Bộ dữ liệu chuẩn Hugging Face (`hllj/vi_gsm8k`)**: Đánh giá suy luận số đa bước quy mô lớn (toàn bộ 1.319 câu hoặc giới hạn với `--limit`).

In [18]:
# 4.1. Benchmark với Google Gemini (Mặc định gemini-3.5-flash kèm chuỗi Fail-fast Lite)
!python benchmark.py --dataset local --provider gemini --model gemini-3.5-flash --limit 5 --delay 1.0 --output colab_benchmark_gemini.json

import json
with open("colab_benchmark_gemini.json", "r", encoding="utf-8") as f:
    rep_local = json.load(f)

print("")
print("[*] TÓM TẮT CHỈ SỐ ĐỊNH LƯỢNG (GEMINI 3.5 FLASH FAIL-FAST):")
for k, v in rep_local.get("metrics", {}).items():
    print(f"  • {k}: {v}")


      HỆ THỐNG ĐÁNH GIÁ ĐỊNH LƯỢNG MULTI-AGENT GIẢI TOÁN THCS (BENCHMARK)     
[*] Tập dữ liệu : Tệp cục bộ: /content/drive/MyDrive/mat3399_final-project/data/thcs_problems.json (15 câu hỏi)
[*] LLM Provider: gemini | Model: gemini-3.5-flash
[*] Sandbox Timeout: 3s | Max Correction Attempts: 3
[*] Tổng số câu hỏi kiểm thử: 5
--------------------------------------------------------------------------------

[1/5] Đang giải [thcs_01] (Toán chuyển động)...
    -> [!] Lỗi không xử lý được: Không thể kết nối tới Gemini API sau các lần thử: HTTP 404: {
  "error": {
    "code": 404,
    "message": "models/gemini-1.5-flash is not found for API version v1beta, or is not supported for generateContent. Call ModelService.ListModels to see the list of available models and their supported methods.",
    "status": "NOT_FOUND"
  }
}


[2/5] Đang giải [thcs_02] (Toán chuyển động)...
    -> [!] Lỗi không xử lý được: Lỗi kết nối Gemini API (gemini-3.5-flash): HTTP 429: {
  "error": {
    "code": 429,
    

In [ ]:
# 4.2. Benchmark với OpenRouter DeepSeek-R1 (Mô hình suy luận mã nguồn mở miễn phí)
# Yêu cầu: Cần cấu hình OPENROUTER_API_KEY ở Bước 2
import os
if os.environ.get("OPENROUTER_API_KEY"):
    !python benchmark.py --dataset local --provider openrouter --model deepseek/deepseek-r1:free --limit 3 --delay 1.0 --output colab_benchmark_openrouter.json
    with open("colab_benchmark_openrouter.json", "r", encoding="utf-8") as f:
        rep_or = json.load(f)
    print("")
    print("[*] TÓM TẮT CHỈ SỐ ĐỊNH LƯỢNG (OPENROUTER DEEPSEEK-R1):")
    for k, v in rep_or.get("metrics", {}).items():
        print(f"  • {k}: {v}")
else:
    print("ℹ️ Bỏ qua benchmark OpenRouter vì chưa cấu hình OPENROUTER_API_KEY.")


In [ ]:
# 4.3. Benchmark trên bộ dữ liệu chuẩn Hugging Face 'hllj/vi_gsm8k'
# Gợi ý: Mặc định chạy 5 câu kiểm thử nhanh. Để chạy toàn bộ 1.319 câu của tập test, hãy bỏ tham số --limit!
!python benchmark.py --dataset vi_gsm8k --provider gemini --limit 5 --delay 1.0 --output colab_benchmark_vi_gsm8k.json

import json
with open("colab_benchmark_vi_gsm8k.json", "r", encoding="utf-8") as f:
    rep_gsm8k = json.load(f)

print("")
print("=" * 65)
print(f"[*] KẾT QUẢ BENCHMARK TRÊN TẬP: {rep_gsm8k.get('dataset')}")
print("=" * 65)
for k, v in rep_gsm8k.get("metrics", {}).items():
    print(f"  • {k}: {v}")


## 🌐 Bước 5: Khởi chạy Giao diện Tương tác Streamlit trên Google Colab
Có 2 phương án mở đường hầm trực tuyến truy cập Streamlit:

### 🟢 Phương án A: Dùng Localtunnel (Miễn phí 100%, không cần tài khoản)
Chạy ô dưới đây. Nhấp vào đường link `*.loca.lt` và nhập mã **IP mật khẩu** được in ra màn hình.

In [ ]:
import urllib.request

# Lấy IP công khai làm mật khẩu truy cập localtunnel
try:
    public_ip = urllib.request.urlopen("https://api.ipify.org").read().decode("utf-8").strip()
except Exception:
    public_ip = urllib.request.urlopen("https://ipv4.icanhazip.com").read().decode("utf-8").strip()

print("=" * 65)
print(f"🔑 MẬT KHẨU ĐĂNG NHẬP (ENDPOINT IP) LÀ: {public_ip}")
print("👉 Khi trang web mở ra, hãy dán IP trên vào ô \x27Endpoint IP\x27 và bấm Click to Submit!")
print("=" * 65)

!npm install -g localtunnel > /dev/null 2>&1
!streamlit run app.py --server.port 8501 > /dev/null 2>&1 & npx localtunnel --port 8501


### 🔵 Phương án B: Dùng Ngrok (Cực kỳ ổn định và mượt mà)
Nếu bạn có tài khoản [ngrok](https://ngrok.com/), nhập Authtoken để mở đường hầm tốc độ cao.

In [17]:
import os
from pyngrok import ngrok
from getpass import getpass

# 1. Đóng sạch các đường hầm Ngrok cũ và giải phóng cổng 8501 nếu có tiến trình Streamlit cũ
try:
    ngrok.kill()
except Exception:
    pass
!pkill -f streamlit > /dev/null 2>&1 || true

# 2. Lấy hoặc nhập NGROK_AUTHTOKEN
ngrok_token = os.getenv("NGROK_AUTHTOKEN", "").strip()
if not ngrok_token:
    ngrok_token = getpass("Nhập NGROK_AUTHTOKEN của bạn (bỏ trống nếu không dùng): ").strip()

if ngrok_token:
    # 3. Cấu hình Authtoken có bắt ngoại lệ chống lỗi "already set" khi chạy lại
    try:
        ngrok.set_auth_token(ngrok_token)
    except Exception as e:
        if "already set" not in str(e).lower():
            print(f"[*] Lưu ý cấu hình token: {e}")

    # 4. Mở đường hầm trực tuyến tới cổng 8501
    try:
        tunnel = ngrok.connect(8501)
        public_url = tunnel.public_url if hasattr(tunnel, "public_url") else str(tunnel)
        print("=" * 65)
        print(f"🚀 Link truy cập Streamlit qua Ngrok: {public_url}")
        print("=" * 65)
        
        # 5. Khởi chạy giao diện web Streamlit
        !streamlit run app.py --server.port 8501
    except Exception as err:
        print(f"❌ Lỗi khi mở đường hầm Ngrok: {err}")
else:
    print("ℹ️ Bỏ qua Ngrok. Hãy dùng Phương án A (Localtunnel) ở trên.")

^C
🚀 Link truy cập Streamlit qua Ngrok: https://denial-circling-overvalue.ngrok-free.dev


2026-11-19 12:55:40.578 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.183.81:8501



  Stopping...


In [15]:
from pyngrok import ngrok
from getpass import getpass

ngrok_token = os.getenv("NGROK_AUTHTOKEN", "")
if not ngrok_token:
    ngrok_token = getpass("Nhập NGROK_AUTHTOKEN của bạn (bỏ trống nếu không dùng): ")

if ngrok_token.strip():
    ngrok.set_auth_token(ngrok_token.strip())
    public_url = ngrok.connect(8501)
    print(f"🚀 Link truy cập Streamlit qua Ngrok: {public_url}")
    !streamlit run app.py --server.port 8501
else:
    print("Bỏ qua Ngrok. Hãy dùng Phương án A (Localtunnel) ở trên.")


PyngrokNgrokError: An error occurred when saving the auth token: b'Authtoken already set in configuration file: /root/.config/ngrok/ngrok.yml\n'